# Algorithms for massive data Project

## Data loading

In [2]:
import os
import zipfile
os.environ['KAGGLE_USERNAME'] = "xxxx"
os.environ['KAGGLE_KEY'] = "xxxx"
!kaggle datasets download -d Cornell-University/arxiv
with zipfile.ZipFile("arxiv.zip", "r") as zip_ref:
    zip_ref.extractall("arxiv_data")

Dataset URL: https://www.kaggle.com/datasets/Cornell-University/arxiv
License(s): CC0-1.0




  0%|          | 0.00/1.63G [00:00<?, ?B/s]
  0%|          | 1.00M/1.63G [00:00<18:50, 1.55MB/s]
  0%|          | 2.00M/1.63G [00:00<12:11, 2.39MB/s]
  0%|          | 3.00M/1.63G [00:01<09:32, 3.05MB/s]
  0%|          | 4.00M/1.63G [00:01<07:30, 3.88MB/s]
  0%|          | 5.00M/1.63G [00:01<07:07, 4.09MB/s]
  0%|          | 6.00M/1.63G [00:01<06:05, 4.78MB/s]
  0%|          | 7.00M/1.63G [00:01<05:51, 4.96MB/s]
  0%|          | 8.00M/1.63G [00:02<05:09, 5.64MB/s]
  1%|          | 9.00M/1.63G [00:02<04:59, 5.82MB/s]
  1%|          | 10.0M/1.63G [00:02<04:34, 6.34MB/s]
  1%|          | 11.0M/1.63G [00:02<04:42, 6.17MB/s]
  1%|          | 12.0M/1.63G [00:02<04:10, 6.94MB/s]
  1%|          | 13.0M/1.63G [00:02<04:18, 6.72MB/s]
  1%|          | 14.0M/1.63G [00:02<04:19, 6.71MB/s]
  1%|          | 15.0M/1.63G [00:03<04:06, 7.04MB/s]
  1%|          | 16.0M/1.63G [00:03<04:20, 6.65MB/s]
  1%|          | 17.0M/1.63G [00:03<04:03, 7.12MB/s]
  1%|          | 18.0M/1.63G [00:03<04:16, 6.75MB/s]
 

## Configuration

In [3]:
import json
DATA_FILE = "arxiv_data/arxiv-metadata-oai-snapshot.json"
MAX_RECORDS = 100000
MIN_PAPERS  = 2
DAMPING = 0.85
TOP_N = 20

## Parsing author names

In [4]:
def get_authors(record):
    parsed = record.get("authors_parsed") or []
    if parsed:
        names = []
        for entry in parsed:
            last = entry[0].strip() if len(entry) > 0 else ""
            first = entry[1].strip() if len(entry) > 1 else ""
            if last:
                names.append(f"{last}, {first}" if first else last)
        return names

    return [a.strip() for a in record.get("authors", "").split(",") if a.strip()]

## Authors count

In [5]:
paper_count = {}
n = 0

with zipfile.ZipFile("arxiv.zip", 'r') as z:
    with z.open("arxiv-metadata-oai-snapshot.json") as f:
        for line in f:
            if n >= MAX_RECORDS:
                break
            record = json.loads(line)
            for author in get_authors(record):
                paper_count[author] = paper_count.get(author, 0) + 1
            n += 1
eligible_authors = {author for author, count in paper_count.items() if count >= MIN_PAPERS}
print(f"Total unique authors: {len(paper_count):,}")
print(f"Eligible authors (written >= {MIN_PAPERS} papers): {len(eligible_authors):,}")

Total unique authors: 135,418
Eligible authors (written >= 2 papers): 60,679


## Co-authorship graph

In [6]:
import networkx as nx
import itertools
from collections import defaultdict

edge_weights = defaultdict(int)
n = 0

with zipfile.ZipFile("arxiv.zip", 'r') as z:
    with z.open("arxiv-metadata-oai-snapshot.json") as f:
        for line in f:
            if n >= MAX_RECORDS:
                break
            record = json.loads(line)
            authors = [a for a in get_authors(record) if a in eligible_authors]
            
            for u, v in itertools.combinations(sorted(authors), 2):
              edge_weights[(u, v)] += 1
            
            n += 1

G = nx.Graph()

for (u, v), w in edge_weights.items():
    G.add_edge(u, v, weight=w)
print(f"Graph with {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges.")

Graph with 56,595 nodes and 490,844 edges.


## PageRank computation

In [10]:
!pip install pandas
!pip install scipy
import scipy as sp
import pandas as pd

pr_unweighted = nx.pagerank(G, alpha=DAMPING, weight=None)
pr_weighted   = nx.pagerank(G, alpha=DAMPING, weight="weight")

# clean pandas df
df = pd.DataFrame({
    "author": list(G.nodes()),
    "papers": [paper_count[node] for node in G.nodes()],
    "degree": [G.degree(node) for node in G.nodes()],
    "pr_unweighted": [pr_unweighted[node] for node in G.nodes()],
    "pr_weighted": [pr_weighted[node] for node in G.nodes()],
})

print(f"\n Top {TOP_N} by Weighted PageRank")
top_weighted = df.nlargest(TOP_N, "pr_weighted")
print(top_weighted[["author", "papers", "degree", "pr_weighted"]].to_string(index=False))


 Top 20 by Weighted PageRank
              author  papers  degree  pr_weighted
    Poor, H. Vincent     105      72     0.000294
Schneider, Donald P.      38     296     0.000183
         Gehrels, N.      63     467     0.000175
      Guo, Guang-Can      42      54     0.000156
        Nori, Franco      48      60     0.000153
       Sarma, S. Das      64      60     0.000149
   Kevrekidis, P. G.      43      54     0.000147
     Canfield, P. C.      44     143     0.000145
         Wang, N. L.      46     157     0.000141
         Chen, G. F.      43     152     0.000137
            Udry, S.      47     242     0.000134
        Peres, Yuval      24      31     0.000133
          Queloz, D.      43     216     0.000132
          Luo, J. L.      41     144     0.000125
          Bouchy, F.      37     238     0.000124
        Henning, Th.      39     133     0.000123
           Zhang, Y.      35     303     0.000123
    Rich, R. Michael      37     128     0.000121
      Osborne, J. P.